# TF-IDF From Scratch

In this notebook we build TF-IDF step by step using only Python and pandas.
No machine learning library is used until the very end (for comparison).

## What is TF-IDF?

Imagine you have 4 product reviews and you want to find the most *important* words in each review.

- The word **"is"** appears in every review. It tells you nothing special. Low importance.
- The word **"excellent"** appears in only one review. It is a strong signal for that review. High importance.

TF-IDF captures this idea:

> **TF-IDF = How often a word appears in THIS document x How rare that word is across ALL documents**

| Step | Name | Question it answers |
|------|------|---------------------|
| 1 | TF (Term Frequency) | How often does this word appear in this document? |
| 2 | IDF (Inverse Document Frequency) | How rare is this word across all documents? |
| 3 | TF-IDF | High frequency + high rarity = important word |

## The Dataset

We use 4 short product reviews. Our goal is to find which words best describe each review.

In [ ]:
import pandas as pd
import math

docs = [
    "Battery life is great camera quality is good",
    "Battery drains fast phone heats while gaming",
    "Camera is excellent in low light battery is decent",
    "Phone design is good performance is smooth"
]

doc_ids = ["D1", "D2", "D3", "D4"]

pd.DataFrame({"doc_id": doc_ids, "text": docs})

## Step 1 — Tokenization

We split each review into individual words and convert everything to lowercase.
This makes sure `Battery` and `battery` are treated as the same word.

In [ ]:
tokenized_docs = [doc.lower().split() for doc in docs]

for doc_id, tokens in zip(doc_ids, tokenized_docs):
    print(f"{doc_id}: {tokens}")

## Step 2 — Build the Vocabulary

We collect every unique word across all 4 documents.
This is our vocabulary — the complete list of words we will score.

In [ ]:
vocab = sorted(set(word for doc in tokenized_docs for word in doc))
print(f"Total unique words: {len(vocab)}")
print(vocab)

## Step 3 — Term Frequency (TF)

**Question: How many times does each word appear in each document?**

We simply count raw occurrences.

Examples from D1 (`Battery life is great camera quality is good`):
- `battery` appears 1 time → TF = 1
- `is` appears 2 times → TF = 2
- `gaming` appears 0 times → TF = 0

We build a table: **rows = documents, columns = words, values = counts**.

In [ ]:
tf_rows = []

for doc_tokens, doc_id in zip(tokenized_docs, doc_ids):
    row = {word: doc_tokens.count(word) for word in vocab}
    row["doc_id"] = doc_id
    tf_rows.append(row)

df_TF = pd.DataFrame(tf_rows).set_index("doc_id")
df_TF

## Step 4 — Document Frequency (DF)

**Question: In how many documents does each word appear at least once?**

- `battery` appears in D1, D2, D3 → DF = 3 (common word)
- `excellent` appears only in D3 → DF = 1 (rare word)

A high DF means the word is common and not very useful for distinguishing documents.
A low DF means the word is specific and very useful.

In [ ]:
df_dict = {}

for word in vocab:
    df_dict[word] = (df_TF[word] > 0).sum()  # count docs where word appears

df_DF = pd.DataFrame.from_dict(df_dict, orient="index", columns=["DF"])
df_DF

## Step 5 — Inverse Document Frequency (IDF)

**Question: How rare is each word across all documents?**

Formula:

$$
IDF(word) = \log\left(\frac{N}{DF(word)}\right)
$$

Where N = total number of documents (4 in our case).

**What this means:**

| Scenario | IDF value | Meaning |
|----------|-----------|---------|
| Word appears in all 4 docs | log(4/4) = 0 | Not useful at all |
| Word appears in 2 docs | log(4/2) = 0.69 | Somewhat useful |
| Word appears in only 1 doc | log(4/1) = 1.39 | Very useful |

Rare words get a **high IDF**. Common words get a **low IDF** (close to 0).

In [ ]:
N = len(doc_ids)

idf_dict = {word: math.log(N / df_val) for word, df_val in df_dict.items()}

df_IDF = pd.DataFrame.from_dict(idf_dict, orient="index", columns=["IDF"])
df_IDF

## Step 6 — TF-IDF Score

Now we multiply TF and IDF together:

$$
TF\text{-}IDF(word, doc) = TF(word, doc) \times IDF(word)
$$

**What this gives us:**

- A word that appears **many times in a document** AND is **rare across all documents** → **high TF-IDF** → very important
- A word like `is` that appears everywhere → low IDF → low TF-IDF even if it appears many times → not important

Each row = document. Each column = word. The value = how important that word is for that document.

In [ ]:
df_TFIDF = df_TF.copy().astype(float)

for word in vocab:
    df_TFIDF[word] = df_TF[word] * df_IDF.loc[word, "IDF"]

df_TFIDF.round(3)

## Step 7 — Top Words Per Document

For each document, we find the words with the highest TF-IDF score.
These are the words that best describe that document.

In [ ]:
for doc_id in df_TFIDF.index:
    row = df_TFIDF.loc[doc_id]
    top_words = row[row > 0].sort_values(ascending=False).head(3)
    print(f"\n{doc_id} — top 3 words:")
    for word, score in top_words.items():
        print(f"  {word:15s}  TF-IDF = {score:.3f}")

## Step 8 — Compare With sklearn

sklearn's `TfidfVectorizer` also computes TF-IDF but with two differences:

1. It removes very common English words like `is`, `in`, `while` (called **stop words**)
2. It **normalizes** each row so all values are between 0 and 1

The rankings should still match closely.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(docs)

df_sklearn = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=doc_ids,
    columns=vectorizer.get_feature_names_out()
)

df_sklearn.round(3)

## Step 9 — Document Similarity (Cosine Similarity)

Now every document is a **vector of numbers** (its TF-IDF scores).
We can measure how similar two documents are by computing the angle between their vectors.

This is called **cosine similarity**:
- Score **1.0** = documents are identical in content
- Score **0.0** = documents share no important words

This is exactly how a search engine finds documents most relevant to your search query.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(df_TFIDF)

df_sim = pd.DataFrame(similarity_matrix, index=doc_ids, columns=doc_ids)
print("Cosine Similarity (higher = more similar content):")
df_sim.round(3)

## Summary

| Step | Formula | What it measures |
|------|---------|------------------|
| TF | count(word in doc) | How often a word appears in this document |
| IDF | log(N / DF) | How rare a word is across all documents |
| TF-IDF | TF x IDF | How important a word is for a specific document |

**Key insight:**
A word is important for a document if it appears **frequently in that document** but **rarely elsewhere**.

- `is`, `the`, `in` → appear everywhere → low IDF → low TF-IDF → ignored
- `excellent`, `gaming`, `design` → appear rarely → high IDF → high TF-IDF → strong signal